# 世界杯冠军预测 - 数据采集与特征工程

使用 API-Sports (api-football.com) v3 获取 2022 世界杯及近期数据，为预测 2026 世界杯冠军做准备。

## 1. API 配置与数据加载

In [ ]:
import requests
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path

# API 配置
API_KEY = 'f1d6df65ff4f697f9a5b56dfb8824cfe'
HEADERS = {'x-apisports-key': API_KEY}
BASE_URL = 'https://v3.football.api-sports.io'
DATA_DIR = Path('wc2026_raw_data')

def api_get(endpoint, params=None, sleep=0.3):
    """通用 API 请求函数，内置速率控制"""
    if params is None:
        params = {}
    resp = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS, params=params, timeout=15)
    time.sleep(sleep)  # 免费版限制 10次/分钟
    return resp.json()

def check_quota():
    """查看今日剩余请求次数"""
    data = api_get("status", sleep=0)
    print(f"今日已用请求: {data['response']['requests']['current']}/{data['response']['requests']['limit_day']}")

print("API 配置完成")
check_quota()

## 2. 加载已采集的数据

已通过 API 获取并缓存的数据：
- `wc2022_fixtures.json` — 2022 世界杯全部 64 场比赛
- `wc2022_standings.json` — 2022 世界杯小组赛积分榜
- `wc2022_top8_team_stats.json` — 8 强球队详细统计
- `wc2022_top8_players.json` — 8 强球队球员数据
- `wc2022_knockout_predictions.json` — 淘汰赛 API 预测数据
- `wc2022_h2h_data.json` — 关键对决交锋记录
- `key_teams_recent_matches.json` — 强队 2023-2024 国际比赛

In [ ]:
# 加载所有已缓存的数据
def load_json(filename):
    with open(DATA_DIR / filename) as f:
        return json.load(f)

wc2022_fixtures = load_json('wc2022_fixtures.json')
wc2022_standings = load_json('wc2022_standings.json')
wc2022_top8_stats = load_json('wc2022_top8_team_stats.json')
wc2022_top8_players = load_json('wc2022_top8_players.json')
wc2022_knockout_preds = load_json('wc2022_knockout_predictions.json')
wc2022_h2h = load_json('wc2022_h2h_data.json')
recent_matches = load_json('key_teams_recent_matches.json')
api_sports_team_ids = json.load(open('api_sports_team_ids.json'))

print("=== 已加载数据概览 ===")
print(f"2022 世界杯比赛: {len(wc2022_fixtures)} 场")
print(f"小组赛积分榜: {len(wc2022_standings)} 个组")
print(f"8强球队统计: {len(wc2022_top8_stats)} 支球队")
print(f"8强球员数据: {len(wc2022_top8_players)} 支球队")
print(f"淘汰赛预测: {len(wc2022_knockout_preds)} 场")
print(f"H2H 交锋记录: {len(wc2022_h2h)} 组对决")
print(f"近期比赛: {len(recent_matches)} 支球队")

# 2026 世界杯小组赛赛程（来自已有的 football-data.org 数据）
teams_2026 = []
for f in DATA_DIR.glob('team_*.json'):
    if f.name.startswith('team_'):
        data = json.load(open(f))
        if data.get('matches'):
            teams_2026.append(data)
print(f"2026 世界杯参赛队: {len(teams_2026)} 支")

## 3. 构建 2022 世界杯特征表

从 2022 世界杯数据中提取每支球队的特征，作为预测模型的基础训练数据。

In [ ]:
# === 3.1 从比赛数据提取球队特征 ===
from collections import defaultdict

def extract_team_features_from_fixtures(fixtures):
    """从比赛列表中提取每支球队的统计特征"""
    team_stats = defaultdict(lambda: {
        'matches_played': 0, 'wins': 0, 'draws': 0, 'losses': 0,
        'goals_for': 0, 'goals_against': 0,
        'clean_sheets': 0,
        'goals_first_half_for': 0, 'goals_first_half_against': 0,
        'goals_second_half_for': 0, 'goals_second_half_against': 0,
        'stage_reached': 'GROUP_STAGE',  # 最深阶段
    })
    
    stage_order = ['GROUP_STAGE', 'KNOWCKOUT_STAGE', 'QUARTER_FINAL', 'SEMI_FINAL', 'FINAL', 'THIRD_PLACE']
    stage_rank = {s: i for i, s in enumerate(stage_order)}
    
    for fx in fixtures:
        home = fx['teams']['home']['name']
        away = fx['teams']['away']['name']
        gh = fx['goals']['home']
        ga = fx['goals']['away']
        stage = fx['league'].get('round', '')
        
        # 更新最深阶段
        for team in [home, away]:
            current = stage_rank.get(team_stats[team]['stage_reached'], 0)
            for s, r in stage_rank.items():
                if s.upper() in stage.upper() and r > current:
                    team_stats[team]['stage_reached'] = s
                    break
        
        # 统计
        for team, gf, ga_val, is_home in [(home, gh, ga, True), (away, ga, gh, False)]:
            ts = team_stats[team]
            ts['matches_played'] += 1
            ts['goals_for'] += gf
            ts['goals_against'] += ga_val
            if ga_val == 0:
                ts['clean_sheets'] += 1
            if gf > ga_val:
                ts['wins'] += 1
            elif gf == ga_val:
                ts['draws'] += 1
            else:
                ts['losses'] += 1
    
    # 派生特征
    for team, ts in team_stats.items():
        ts['goal_diff'] = ts['goals_for'] - ts['goals_against']
        ts['goals_per_match'] = round(ts['goals_for'] / max(ts['matches_played'], 1), 2)
        ts['goals_conceded_per_match'] = round(ts['goals_against'] / max(ts['matches_played'], 1), 2)
        ts['win_rate'] = round(ts['wins'] / max(ts['matches_played'], 1), 3)
        ts['clean_sheet_rate'] = round(ts['clean_sheets'] / max(ts['matches_played'], 1), 3)
    
    return dict(team_stats)

team_features_2022 = extract_team_features_from_fixtures(wc2022_fixtures)

# 转为 DataFrame
df_team_features = pd.DataFrame(team_features_2022).T
df_team_features.index.name = 'team'
df_team_features = df_team_features.sort_values('wins', ascending=False)
print("=== 2022 世界杯球队特征（按胜场排序） ===")
print(df_team_features[['matches_played', 'wins', 'draws', 'losses', 'goals_for', 'goals_against', 'goal_diff', 'win_rate', 'stage_reached']].to_string())

In [ ]:
# === 3.2 从积分榜数据补充小组赛特征 ===
def extract_standings_features(standings_data):
    """从积分榜数据提取小组赛特征"""
    rows = []
    for group_data in standings_data:
        for group_table in group_data['league']['standings']:
            for entry in group_table:
                team = entry['team']['name']
                rows.append({
                    'team': team,
                    'group': entry.get('group', ''),
                    'rank': entry['rank'],
                    'points': entry['points'],
                    'form': entry.get('form', ''),
                    'group_goals_for': entry['all']['goals']['for'],
                    'group_goals_against': entry['all']['goals']['against'],
                    'group_goal_diff': entry['goalsDiff'],
                    'group_win_rate': round(entry['all']['win'] / max(entry['all']['played'], 1), 3),
                })
    return pd.DataFrame(rows)

df_standings = extract_standings_features(wc2022_standings)
print("=== 小组赛积分榜特征 ===")
print(df_standings.sort_values('rank').to_string())

In [ ]:
# === 3.3 从球队详细统计提取高级特征（8强球队） ===
def extract_advanced_stats(team_stats_dict):
    """从 API teams/statistics 提取高级特征"""
    rows = []
    for tid, stats in team_stats_dict.items():
        team_name = stats['team']['name']
        lineup = stats.get('lineups', [{}])[0] if stats.get('lineups') else {}
        
        # 进球分布
        goals_data = stats.get('goals', {}).get('for', {}).get('total', {})
        conceding_data = stats.get('goals', {}).get('against', {}).get('total', {})
        
        # 传控数据
        passes = stats.get('passes', {})
        possession = stats.get('goals', {}).get('for', {}).get('average', {})
        
        row = {
            'team': team_name,
            'formation': lineup.get('formation'),
            'clean_sheets_total': stats.get('clean_sheet', {}).get('total', 0),
            'clean_sheet_rate': stats.get('clean_sheet', {}).get('percentage', '0%'),
            'failed_to_score': stats.get('failed_to_score', {}).get('total', 0),
            'avg_goals_scored': stats.get('goals', {}).get('for', {}).get('average', {}).get('total', 0),
            'avg_goals_conceded': stats.get('goals', {}).get('against', {}).get('average', {}).get('total', 0),
            'shots_on_target_per_game': stats.get('shots', {}).get('on', {}).get('average', 0),
            'shots_total_per_game': stats.get('shots', {}).get('total', {}).get('average', 0),
            'corners_per_game': stats.get('corners', {}).get('average', 0),
            'fouls_per_game': stats.get('fouls', {}).get('average', 0),
            'offsides_per_game': stats.get('offsides', {}).get('average', 0),
            'yellow_cards': stats.get('cards', {}).get('yellow', 0),
            'red_cards': stats.get('cards', {}).get('red', 0),
            'total_passes_avg': passes.get('total', {}).get('average', 0),
            'accurate_passes_avg': passes.get('accurate', {}).get('average', 0),
            'pass_accuracy': passes.get('accuracy', '0%'),
        }
        rows.append(row)
    return pd.DataFrame(rows)

df_advanced = extract_advanced_stats(wc2022_top8_stats)
print("=== 8强球队高级统计特征 ===")
print(df_advanced.to_string())

In [ ]:
# === 3.4 从球员数据提取球队阵容特征 ===
def extract_squad_features(players_data):
    """从球员数据提取球队阵容质量特征"""
    rows = []
    for team_name, players in players_data.items():
        if not players:
            continue
        total_rating = 0
        rating_count = 0
        positions = defaultdict(int)
        ages = []
        
        for p in players:
            player = p['player']
            statistics = p.get('statistics', [{}])[0] if p.get('statistics') else {}
            
            rating = statistics.get('games', {}).get('rating', 0)
            if rating and rating != '0.0':
                total_rating += float(rating)
                rating_count += 1
            
            pos = statistics.get('games', {}).get('position', 'Unknown')
            positions[pos] += 1
            
            if player.get('age'):
                ages.append(player['age'])
        
        rows.append({
            'team': team_name,
            'squad_size': len(players),
            'avg_player_rating': round(total_rating / max(rating_count, 1), 2),
            'rated_players': rating_count,
            'avg_age': round(np.mean(ages), 1) if ages else 0,
            'positions': dict(positions),
        })
    return pd.DataFrame(rows)

df_squad = extract_squad_features(wc2022_top8_players)
print("=== 球队阵容质量特征 ===")
print(df_squad.to_string())

## 4. 构建 2026 世界杯预测特征

基于 2026 世界杯分组赛程 + 历史数据，构建预测模型所需的特征。

In [ ]:
# === 4.1 解析 2026 世界杯分组 ===
from collections import defaultdict

groups_2026 = defaultdict(list)
all_teams_2026 = set()

for team_data in teams_2026:
    team_name = team_data['team']['name']
    all_teams_2026.add(team_name)
    for match in team_data['matches']:
        group = match.get('group', '')
        if group:
            groups_2026[group].append(team_name)
            break  # 每队只需记录一次分组

print("=== 2026 世界杯分组 ===")
for group, teams in sorted(groups_2026.items()):
    print(f"{group}: {', '.join(sorted(teams))}")
print(f"\n总计: {len(all_teams_2026)} 支参赛球队")

In [ ]:
# === 4.2 为 2026 球队匹配 2022 历史特征 ===

# 球队名称映射（2026 数据 vs 2022 API-Sports 数据中可能的名称差异）
name_mapping = {
    'United States': 'USA',
    'South Korea': 'South Korea',
    'Curaçao': None,  # 未参加 2022 世界杯
    'Haiti': None,
    'South Africa': None,
    'Cape Verde Islands': None,
    'Ivory Coast': 'Ivory Coast',
    'Norway': None,
    'Uzbekistan': None,
    'Jordan': None,
    'Panama': None,
    'Austria': None,
    'Algeria': None,
    'New Zealand': None,
}

# 构建 2026 球队特征表
def get_2022_features(team_name):
    """查找球队在 2022 世界杯的特征"""
    # 先查原始名
    if team_name in team_features_2022:
        return team_features_2022[team_name]
    # 再查映射
    mapped = name_mapping.get(team_name)
    if mapped and mapped in team_features_2022:
        return team_features_2022[mapped]
    return None

rows_2026 = []
for team_name in sorted(all_teams_2026):
    features_2022 = get_2022_features(team_name)
    row = {
        'team': team_name,
        'played_2022': features_2022['matches_played'] if features_2022 else 0,
        'wins_2022': features_2022['wins'] if features_2022 else 0,
        'losses_2022': features_2022['losses'] if features_2022 else 0,
        'goals_for_2022': features_2022['goals_for'] if features_2022 else 0,
        'goals_against_2022': features_2022['goals_against'] if features_2022 else 0,
        'goal_diff_2022': features_2022['goal_diff'] if features_2022 else 0,
        'win_rate_2022': features_2022['win_rate'] if features_2022 else 0,
        'clean_sheet_rate_2022': features_2022['clean_sheet_rate'] if features_2022 else 0,
        'stage_reached_2022': features_2022['stage_reached'] if features_2022 else 'N/A',
    }
    
    # 补充积分榜数据
    standing_row = df_standings[df_standings['team'] == team_name]
    if standing_row.empty:
        mapped = name_mapping.get(team_name)
        if mapped:
            standing_row = df_standings[df_standings['team'] == mapped]
    if not standing_row.empty:
        s = standing_row.iloc[0]
        row['group_rank_2022'] = s['rank']
        row['group_points_2022'] = s['points']
        row['group_win_rate_2022'] = s['group_win_rate']
    else:
        row['group_rank_2022'] = None
        row['group_points_2022'] = None
        row['group_win_rate_2022'] = None
    
    # 补充高级统计（仅 8 强球队）
    adv_row = df_advanced[df_advanced['team'] == team_name]
    if not adv_row.empty:
        a = adv_row.iloc[0]
        row['avg_goals_scored_2022'] = a.get('avg_goals_scored', 0)
        row['avg_goals_conceded_2022'] = a.get('avg_goals_conceded', 0)
        row['shots_on_target_avg_2022'] = a.get('shots_on_target_per_game', 0)
        row['pass_accuracy_2022'] = a.get('pass_accuracy', '0%')
    else:
        row['avg_goals_scored_2022'] = None
        row['avg_goals_conceded_2022'] = None
        row['shots_on_target_avg_2022'] = None
        row['pass_accuracy_2022'] = None
    
    # 从近期比赛计算近期状态（2023-2024）
    team_recent = recent_matches.get(team_name, [])
    if not team_recent:
        mapped = name_mapping.get(team_name)
        if mapped:
            team_recent = recent_matches.get(mapped, [])
    
    if team_recent:
        recent_wins = sum(1 for m in team_recent 
                         if m['goals']['home'] is not None and 
                         ((m['teams']['home']['name'] == team_name and m['goals']['home'] > m['goals']['away']) or
                          (m['teams']['away']['name'] == team_name and m['goals']['away'] > m['goals']['home'])))
        total_played = len([m for m in team_recent if m['goals']['home'] is not None])
        row['recent_match_count'] = total_played
        row['recent_win_rate'] = round(recent_wins / max(total_played, 1), 3)
    else:
        row['recent_match_count'] = 0
        row['recent_win_rate'] = 0
    
    rows_2026.append(row)

df_2026_features = pd.DataFrame(rows_2026).set_index('team')
print("=== 2026 世界杯球队特征表 ===")
print(df_2026_features.sort_values('win_rate_2022', ascending=False).to_string())

In [ ]:
# === 4.3 保存特征表 ===
# 保存合并后的特征表为 CSV，方便后续建模使用
df_2026_features.to_csv('wc2026_team_features.csv')
df_team_features.to_csv('wc2022_all_team_features.csv')
df_advanced.to_csv('wc2022_advanced_stats.csv')
df_squad.to_csv('wc2022_squad_features.csv')

print("特征表已保存:")
print(f"  wc2026_team_features.csv — {len(df_2026_features)} 支 2026 参赛球队")
print(f"  wc2022_all_team_features.csv — {len(df_team_features)} 支 2022 参赛球队")
print(f"  wc2022_advanced_stats.csv — {len(df_advanced)} 支 2022 八强球队")
print(f"  wc2022_squad_features.csv — {len(df_squad)} 支球队阵容数据")

## 5. 基于 ELO 模型的冠军预测（基础版）

使用 ELO 评分系统，基于 2022 世界杯和近期比赛结果，预测 2026 世界杯冠军。

**方法说明：**
1. 用 2022 世界杯全部 64 场比赛作为训练数据，初始化各队 ELO 分
2. 用 2023-2024 国际比赛（友谊赛、预选赛、欧洲杯/美洲杯等）更新 ELO
3. 模拟 2026 世界杯小组赛 + 淘汰赛，用 ELO 概率决定胜负
4. 多次蒙特卡洛模拟，统计冠军概率

In [ ]:
# === 5.1 ELO 评分系统实现 ===
import random

class EloRating:
    """ELO 评分系统"""
    
    def __init__(self, K=32, home_advantage=65, initial_rating=1500):
        self.K = K  # 评分变化幅度
        self.home_advantage = home_advantage  # 主场优势（ELO分）
        self.ratings = {}
        self.initial_rating = initial_rating
    
    def get_rating(self, team):
        return self.ratings.get(team, self.initial_rating)
    
    def expected_score(self, rating_a, rating_b):
        """计算预期得分概率"""
        return 1.0 / (1.0 + 10 ** ((rating_b - rating_a) / 400))
    
    def predict(self, team_a, team_b, is_neutral=True):
        """预测比赛结果，返回 (主队胜率, 平率, 客队胜率)"""
        ra = self.get_rating(team_a)
        rb = self.get_rating(team_b)
        if not is_neutral:
            ra += self.home_advantage
        
        ea = self.expected_score(ra, rb)
        eb = self.expected_score(rb, ra)
        
        # 世界杯淘汰赛无平局（有加时赛/点球），小组赛有平局
        win_a = ea
        win_b = eb
        
        return win_a, win_b
    
    def update(self, team_a, team_b, goals_a, goals_b, is_neutral=True):
        """根据比赛结果更新 ELO 评分"""
        ra = self.get_rating(team_a)
        rb = self.get_rating(team_b)
        if not is_neutral:
            ra += self.home_advantage
        
        ea = self.expected_score(ra, rb)
        eb = 1 - ea
        
        # 实际得分: 胜=1, 平=0.5, 负=0
        if goals_a > goals_b:
            sa, sb = 1.0, 0.0
        elif goals_a == goals_b:
            sa, sb = 0.5, 0.5
        else:
            sa, sb = 0.0, 1.0
        
        # 更新评分
        self.ratings[team_a] = ra + self.K * (sa - ea) - (self.home_advantage if not is_neutral else 0)
        self.ratings[team_b] = rb + self.K * (sb - eb)
        
        return self.ratings[team_a], self.ratings[team_b]

# 初始化 ELO 系统并用 2022 世界杯数据训练
elo = EloRating(K=40, home_advantage=0)  # 世界杯大多在中立场

# 用 2022 世界杯比赛训练
for fx in wc2022_fixtures:
    home = fx['teams']['home']['name']
    away = fx['teams']['away']['name']
    gh = fx['goals']['home']
    ga = fx['goals']['away']
    if gh is not None and ga is not None:
        elo.update(home, away, gh, ga)

# 用 2023-2024 近期比赛更新
for team_name, matches in recent_matches.items():
    for m in matches:
        home = m['teams']['home']['name']
        away = m['teams']['away']['name']
        gh = m['goals']['home']
        ga = m['goals']['away']
        if gh is not None and ga is not None:
            # 非世界杯比赛可能非中立场地，但国际赛会大部分为中立
            elo.update(home, away, gh, ga)

# 显示 ELO 排名
elo_ranking = sorted(elo.ratings.items(), key=lambda x: -x[1])
print("=== ELO 排名（2022 WC + 近期比赛后） ===")
print(f"{'排名':<4} {'球队':<20} {'ELO':>6}")
print("-" * 32)
for i, (team, rating) in enumerate(elo_ranking[:20], 1):
    print(f"{i:<4} {team:<20} {rating:>6.0f}")

In [ ]:
# === 5.2 蒙特卡洛模拟 2026 世界杯 ===
def simulate_group_stage(groups, elo):
    """模拟小组赛，返回每组前2名"""
    qualifiers = []
    for group_name, teams in sorted(groups.items()):
        points = {t: 0 for t in teams}
        goal_diff = {t: 0 for t in teams}
        goals_for = {t: 0 for t in teams}
        
        # 每组4队，循环赛（每队打3场）
        for i, t1 in enumerate(teams):
            for t2 in teams[i+1:]:
                p_a, p_b = elo.predict(t1, t2)
                rand = random.random()
                
                # 模拟比分
                if rand < p_a * 0.6:  # 60% 概率分出胜负
                    goals_a = max(1, int(random.gauss(2, 1)))
                    goals_b = max(0, int(random.gauss(0.8, 0.8)))
                    if goals_a <= goals_b:
                        goals_b = goals_a - 1
                elif rand > 1 - p_b * 0.6:
                    goals_b = max(1, int(random.gauss(2, 1)))
                    goals_a = max(0, int(random.gauss(0.8, 0.8)))
                    if goals_b <= goals_a:
                        goals_a = goals_b - 1
                else:
                    goals_a = goals_b = max(0, int(random.gauss(1, 0.7)))
                
                # 更新积分
                if goals_a > goals_b:
                    points[t1] += 3
                elif goals_a < goals_b:
                    points[t2] += 3
                else:
                    points[t1] += 1
                    points[t2] += 1
                goal_diff[t1] += goals_a - goals_b
                goal_diff[t2] += goals_b - goals_a
                goals_for[t1] += goals_a
                goals_for[t2] += goals_b
        
        # 排序: 积分 > 净胜球 > 进球数
        ranked = sorted(teams, key=lambda t: (-points[t], -goal_diff[t], -goals_for[t]))
        qualifiers.append((ranked[0], ranked[1]))
        print(f"  {group_name}: 1st={ranked[0]}({points[ranked[0]]}pts)  2nd={ranked[1]}({points[ranked[1]]}pts)")
    
    return qualifiers

def simulate_knockout(pairs, elo, round_name):
    """模拟淘汰赛一轮"""
    winners = []
    print(f"\n  --- {round_name} ---")
    for t1, t2 in pairs:
        p_a, p_b = elo.predict(t1, t2)
        rand = random.random()
        # 淘汰赛无平局
        if rand < p_a:
            winner = t1
        else:
            winner = t2
        print(f"  {t1} ({p_a:.0%}) vs {t2} ({p_b:.0%}) => {winner}")
        winners.append(winner)
    return winners

def simulate_world_cup(groups, elo):
    """完整模拟一届世界杯，返回冠军"""
    # 小组赛
    qualifiers = simulate_group_stage(groups, elo)
    
    # 16强: 每组第1 vs 另一组第2
    # 简化对阵: A1 vs B2, B1 vs A2, C1 vs D2, D1 vs C2, ...
    r16_pairs = []
    for i in range(0, len(qualifiers), 2):
        r16_pairs.append((qualifiers[i][0], qualifiers[i+1][1]))
        r16_pairs.append((qualifiers[i+1][0], qualifiers[i][0][0] if False else qualifiers[i][1]))
    # 重新按标准赛制
    r16_pairs = []
    q = [team for pair in qualifiers for team in pair]  # [A1,A2,B1,B2,...]
    for i in range(0, len(q), 4):
        r16_pairs.append((q[i], q[i+3]))
        r16_pairs.append((q[i+1], q[i+2]))
    
    # 16强 -> 8强
    qf_teams = simulate_knockout(r16_pairs, elo, "Round of 16")
    
    # 8强 -> 4强
    sf_teams = simulate_knockout([(qf_teams[i], qf_teams[i+1]) for i in range(0, 8, 2)], elo, "Quarter-finals")
    
    # 半决赛
    finalists = simulate_knockout([(sf_teams[0], sf_teams[1]), (sf_teams[2], sf_teams[3])], elo, "Semi-finals")
    
    # 决赛
    champion = simulate_knockout([(finalists[0], finalists[1])], elo, "FINAL")[0]
    
    return champion

# 运行蒙特卡洛模拟
N_SIMULATIONS = 100
champion_counts = defaultdict(int)

print(f"\n{'='*60}")
print(f"开始蒙特卡洛模拟: {N_SIMULATIONS} 次")
print(f"{'='*60}\n")

for sim in range(N_SIMULATIONS):
    random.seed(sim)  # 可复现
    champion = simulate_world_cup(groups_2026, elo)
    champion_counts[champion] += 1
    if (sim + 1) % 20 == 0:
        print(f"\n--- 已完成 {sim+1}/{N_SIMULATIONS} 次模拟 ---")

# 统计结果
print(f"\n{'='*60}")
print(f"2026 世界杯冠军预测结果 (蒙特卡洛模拟 {N_SIMULATIONS} 次)")
print(f"{'='*60}\n")
print(f"{'球队':<20} {'夺冠次数':>8} {'夺冠概率':>8}")
print("-" * 40)
for team, count in sorted(champion_counts.items(), key=lambda x: -x[1]):
    print(f"{team:<20} {count:>8} {count/N_SIMULATIONS:>7.1%}")